In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
billingledgerdetail_table = dbutils.widgets.get("billingledgerdetail_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")
patient_table = dbutils.widgets.get("patient_table")
branch_table = dbutils.widgets.get("branch_table")
payer_table = dbutils.widgets.get("payer_table")
financialclass_table = dbutils.widgets.get("financialclass_table")
payerauthorization_table = dbutils.widgets.get("payerauthorization_table")
assignment_table = dbutils.widgets.get("assignment_table")
assignmentbilling_table = dbutils.widgets.get("assignmentbilling_table")
payerauthorizationsegmentbillingcode_table = dbutils.widgets.get("payerauthorizationsegmentbillingcode_table")
payerbillingcode_table = dbutils.widgets.get("payerbillingcode_table")
billingcode_table = dbutils.widgets.get("billingcode_table")
payerbillingcodemodifier_table = dbutils.widgets.get("payerbillingcodemodifier_table")
billingcodemodifier_table = dbutils.widgets.get("billingcodemodifier_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW charges_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(ChargeID AS STRING) AS ChargeID,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(SvcDate AS STRING) AS SvcDate,
  CAST(PostDate AS STRING) AS PostDate,
  CAST(ChgCode AS STRING) AS ChgCode,
  CAST(ChgDesc AS STRING) AS ChgDesc,
  CAST(ChgDept AS STRING) AS ChgDept,
  NULL AS ChgDeptDesc,
  CAST(RevCode AS STRING) AS RevCode,
  CAST(CPTCode AS STRING) AS CPTCode,
  CAST(Mod1 AS STRING) AS Mod1,
  CAST(Mod2 AS STRING) AS Mod2,
  CAST(Mod3 AS STRING) AS Mod3,
  CAST(Mod4 AS STRING) AS Mod4,
  CAST(ChgQty AS INT) AS ChgQty,
  CAST(UnitAmt AS INT) AS UnitAmt,
  CAST(ChgAmt AS DOUBLE) AS ChgAmt,
  CAST(FinClass AS STRING) AS FinClass,
  NULL AS PrimaryPayorCode,
  NULL AS PrimaryPayorDesc,
  NULL AS PatientType,
  CAST(AuthNbr AS STRING) AS AuthNbr,
  SourceSystemKey AS SourceSystemKey
FROM (
  WITH 
  charges_cte AS (
    SELECT DISTINCT
        CAST('{fetch_date}' AS DATE) AS ReportingDate,
        bld.Id AS ChargeID, --charge id mismatch any other option?
        b.ExternalId AS FacilityCode,
        bl.ClaimNumber AS AcctNbr,
        bld.ServiceDate AS SvcDate,
        bl.DateBilled AS PostDate,
        REPLACE(REPLACE(REPLACE(trim(split(bld.ServiceCodeString, '-')[0]), '\r', ''), '\n', ''), '\t', '') AS ChgCode,
        bld.ServiceCodeString AS ChgDesc,
        CASE bld.BillingRateType 
            WHEN 1 THEN 'Per Hour'
            WHEN 2 THEN 'Per Unit'
            WHEN 3 THEN 'Per Visit'
        END AS ChgDept,
        pbc.RevenueCode AS RevCode,
        bc.Code AS CPTCode,
        bcm1.Code AS Mod1,
        bcm2.Code AS Mod2,
        bcm3.Code AS Mod3,
        bcm4.Code AS Mod4,
        CASE bld.BillingRateType 
            WHEN 1 THEN bld.BillableHours
            WHEN 2 THEN bld.ActualUnits
            WHEN 3 THEN 1
        END AS ChgQty,  -- data mismathces is the condition correct 
        bld.Rate AS UnitAmt,
        bld.TotalAmount AS ChgAmt, -- is the condition correct
        fc.Abbreviation AS FinClass,
        pa_first.AuthorizationNumber AS AuthNbr,
        '19' AS SourceSystemKey
    FROM {source_table} bl
    JOIN {billingledgerdetail_table} bld ON bld.BillingLedgerId = bl.Id
    -- LEFT JOIN to detailif we want to not miss claimnumbers
    -- LEFT JOIN {billingledgerdetail_table} bld ON bld.BillingLedgerId = bl.Id
    JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
    JOIN {patient_table} p ON p.Id = pp.PatientId
    JOIN {branch_table} b ON b.Id = p.BranchId
    JOIN {payer_table} pay ON pay.Id = pp.PayerId
    LEFT JOIN {financialclass_table} fc ON fc.Id = pay.FinancialClassId
    -- Picking one authorization per PatientPayer 
    LEFT JOIN (
        SELECT PatientPayerId, AuthorizationNumber,
              ROW_NUMBER() OVER (PARTITION BY PatientPayerId ORDER BY CreatedDate DESC) as rn
        FROM {payerauthorization_table}
    ) pa_first ON pa_first.PatientPayerId = pp.Id AND pa_first.rn = 1
    -- Picking assignment per patient+date
    JOIN {assignment_table} a
    ON a.PatientId = p.Id
    JOIN {assignmentbilling_table} ab 
    ON ab.AssignmentId = a.Id
    AND DATE(ab.start) >= to_date(split(bl.DatesOfService, ' - ')[0], 'MM/dd/yyyy')
    AND DATE(ab.start) <= to_date(split(bl.DatesOfService, ' - ')[1], 'MM/dd/yyyy')
    AND a.AssignedId IS NOT NULL
    ---
    JOIN {payerauthorizationsegmentbillingcode_table} pasbc 
        ON pasbc.Id = ab.PayerAuthorizationSegmentBillingCodeId
    JOIN {payerbillingcode_table} pbc ON pbc.Id = pasbc.PayerBillingCodeId
    JOIN {billingcode_table} bc ON bc.Id = pbc.BillingCodeId
    -- Modifiers
    LEFT JOIN {payerbillingcodemodifier_table} pbcm1 
        ON pbcm1.PayerBillingCodeId = pbc.Id AND pbcm1.Position = 1
    LEFT JOIN {billingcodemodifier_table} bcm1 
        ON bcm1.Id = pbcm1.ModifierId
    LEFT JOIN {payerbillingcodemodifier_table} pbcm2 
        ON pbcm2.PayerBillingCodeId = pbc.Id AND pbcm2.Position = 2
    LEFT JOIN {billingcodemodifier_table} bcm2 
        ON bcm2.Id = pbcm2.ModifierId
    LEFT JOIN {payerbillingcodemodifier_table} pbcm3 
        ON pbcm3.PayerBillingCodeId = pbc.Id AND pbcm3.Position = 3
    LEFT JOIN {billingcodemodifier_table} bcm3 
        ON bcm3.Id = pbcm3.ModifierId
    LEFT JOIN {payerbillingcodemodifier_table} pbcm4 
        ON pbcm4.PayerBillingCodeId = pbc.Id AND pbcm4.Position = 4
    LEFT JOIN {billingcodemodifier_table} bcm4 
        ON bcm4.Id = pbcm4.ModifierId
    WHERE bl.isActive='true'
    -- WHERE bl.ClaimNumber = '62001FC1971'
    -- WHERE bl.ClaimNumber = '4919FCS1000'

    -- ChargeID mismatch --
    -- ChgQty/ChgAmt differences — Data values differ  Our logic is correct (BillableHours for Per Hour, ActualUnits for Per Unit, TotalAmount for ChgAmt).
    -- Missing 1 row (356379) — The assignment join filters it out because no assignment exists for that service date (2025-02-28).
    -- ~100K claims missing — billingledger has 410K rows but billingledgerdetail only has 305K. Claims without detail records are excluded. 
  ),
  charges_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM charges_cte
  )
  SELECT 
    ReportingDate,
    ChargeID,
    FacilityCode,
    AcctNbr,
    SvcDate,
    PostDate,
    ChgCode,
    ChgDesc,
    ChgDept,
    RevCode,
    CPTCode,
    Mod1,
    Mod2,
    Mod3,
    Mod4,
    ChgQty,
    UnitAmt,
    ChgAmt,
    FinClass,
    AuthNbr,
    SourceSystemKey
  FROM charges_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING charges_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
UPDATE SET
    tgt.ChargeID = src.ChargeID,
    tgt.FacilityCode = src.FacilityCode,
    tgt.SvcDate = src.SvcDate,
    tgt.PostDate = src.PostDate,
    tgt.ChgCode = src.ChgCode,
    tgt.ChgDesc = src.ChgDesc,
    tgt.ChgDept = src.ChgDept,
    tgt.ChgDeptDesc = src.ChgDeptDesc,
    tgt.RevCode = src.RevCode,
    tgt.CPTCode = src.CPTCode,
    tgt.Mod1 = src.Mod1,
    tgt.Mod2 = src.Mod2,
    tgt.Mod3 = src.Mod3,
    tgt.Mod4 = src.Mod4,
    tgt.ChgQty = src.ChgQty,
    tgt.UnitAmt = src.UnitAmt,
    tgt.ChgAmt = src.ChgAmt,
    tgt.FinClass = src.FinClass,
    tgt.PrimaryPayorCode = src.PrimaryPayorCode,
    tgt.PrimaryPayorDesc = src.PrimaryPayorDesc,
    tgt.PatientType = src.PatientType,
    tgt.AuthNbr = src.AuthNbr,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    ChargeID,
    FacilityCode,
    AcctNbr,
    SvcDate,
    PostDate,
    ChgCode,
    ChgDesc,
    ChgDept,
    ChgDeptDesc,
    RevCode,
    CPTCode,
    Mod1,
    Mod2,
    Mod3,
    Mod4,
    ChgQty,
    UnitAmt,
    ChgAmt,
    FinClass,
    PrimaryPayorCode,
    PrimaryPayorDesc,
    PatientType,
    AuthNbr,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.ChargeID,
    src.FacilityCode,
    src.AcctNbr,
    src.SvcDate,
    src.PostDate,
    src.ChgCode,
    src.ChgDesc,
    src.ChgDept,
    src.ChgDeptDesc,
    src.RevCode,
    src.CPTCode,
    src.Mod1,
    src.Mod2,
    src.Mod3,
    src.Mod4,
    src.ChgQty,
    src.UnitAmt,
    src.ChgAmt,
    src.FinClass,
    src.PrimaryPayorCode,
    src.PrimaryPayorDesc,
    src.PatientType,
    src.AuthNbr,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)